In [1]:
import glob
import os
import torch
torch.cuda.empty_cache()
import torchvision
import numpy as np
import matplotlib.pyplot as plt
import cv2
import time
import pickle

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:512"

In [2]:
from torchvision.datasets.vision import VisionDataset
from torch.utils.data import IterableDataset
from torchvision.datasets.video_utils import VideoClips
# from video_clip import VideoClips
import torch.utils.data as data
from bat_seg_models import ThreeLayerSemSegNetWideView, UNET, UNETTraditional
from frame_augmentors import MaskNormalize, Mask3dto2d, AddDim, ToFloat, MaskCompose, MaskToTensor
import bat_functions
from CountLine import CountLine

In [3]:
import matplotlib.pyplot as plt
im_file = "D:/kasanka-bats/30-Oct-2020/BBC/example-frames/30-Oct-2020_BBC_obs-ind_17550.jpg"
#im_file = ".../kasanka-bats/frames/17Nov/card-f/GP039791/GP039791_15948.jpg"
im = plt.imread(im_file)

im.shape

(1516, 2700)

In [4]:
root_output_folder = "H:/kasanka-bats/"
camera_folders = []
drives_to_search = ['E:', 'K:', 'F:']

for drive in drives_to_search:
    day_folders = sorted(glob.glob(os.path.join(drive, 'KasankaCameras', '*')))
    print(day_folders)
    days = [path.split('\\')[-1] for path in day_folders]
    for day in days:
        print(day)
        raw_camera_folders = []
        day_camera_folders = sorted(glob.glob(os.path.join(drive, 'KasankaCameras', day, "*")))  # Change the path here
        raw_camera_folders.extend(day_camera_folders)

        for camera_folder in raw_camera_folders:
            videos = sorted(glob.glob(os.path.join(camera_folder, '*_flipped.[Mm][Pp]4')))  # Update the file pattern here
            if videos:
                camera_name = camera_folder.split('\\')[-1]
                print(camera_name)
                if not os.path.exists(os.path.join(root_output_folder, day, camera_name, 'flipped_centers.npy')):
                    camera_folders.append(camera_folder)
                    print(camera_folder)
                    print('--------------')

camera_folders = list(set(camera_folders))
print(camera_folders)


['E:KasankaCameras\\20221101', 'E:KasankaCameras\\20221116', 'E:KasankaCameras\\20221124', 'E:KasankaCameras\\20221201', 'E:KasankaCameras\\20221213', 'E:KasankaCameras\\20221219']
20221101
9 KK
20221116
3 Chinyangali
20221124
1 Fibwe Parking
5 Puku
20221201
3 Chinyangali
20221213
3 Chinyangali
6 Sunset
8 Musola Path
20221219
3 Chinyangali
5 Puku
6 Sunset
['K:KasankaCameras\\20211026', 'K:KasankaCameras\\20211101', 'K:KasankaCameras\\20211110', 'K:KasankaCameras\\20211116', 'K:KasankaCameras\\20211123', 'K:KasankaCameras\\20211201', 'K:KasankaCameras\\20211207', 'K:KasankaCameras\\20211215', 'K:KasankaCameras\\20211221']
20211026
FibweParking
20211101
20211110
20211116
20211123
20211201
20211207
20211215
20211221
['F:KasankaCameras\\12-Dec-2019', 'F:KasankaCameras\\16-Nov-2020', 'F:KasankaCameras\\16-Oct-2020', 'F:KasankaCameras\\17-Nov-2020', 'F:KasankaCameras\\18-Nov-2020', 'F:KasankaCameras\\2-Dec-2020', 'F:KasankaCameras\\24-Nov-2019', 'F:KasankaCameras\\27-Nov-2019', 'F:KasankaCam

In [5]:
class BatIterableDataset(IterableDataset):
    def __init__(self, video_files, augmentor=None, max_bad_reads=300):
        self.vid_cap = cv2.VideoCapture(video_files[0])
        self.video_files = video_files
        assert self.vid_cap.isOpened()
        self.more_frames = True
        # How many times a frame can come up false 
        # before assuming end of video
        self.max_bad_reads = max_bad_reads
        self.total_frames_read = 0
        self.total_bad_reads = 0
        self.augmentor = augmentor
        self.video_number = 0
        
    def more_videos(self):
        return self.video_number < len(self.video_files)
    
    def start_next_video(self):
        if self.vid_cap.isOpened():
            self.vid_cap.release()
        self.video_number += 1
        if self.video_number < len(self.video_files):
            print('starting new video')
            print(self.get_read_frame_info())
            self.vid_cap = cv2.VideoCapture(self.video_files[self.video_number])
        
    def video_generator(self):
        while(self.vid_cap.isOpened() or self.more_videos()):
            if not self.vid_cap.isOpened():
                self.start_next_video()
            good_read = False
            num_bad_reads = 0
            while (not good_read and (num_bad_reads < self.max_bad_reads)):
                grabbed, frame = self.vid_cap.read()
                if grabbed:
                    good_read = True
                    self.total_frames_read += 1
                    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    frame = {'image': frame[2:-2, 2:-2]}
                    if np.mean(frame['image'][::50,::50, 2]) < 5:
                        print('too dark')
                        self.vid_cap.release()
                        break
                        
                    if self.augmentor:
                        frame = self.augmentor(frame)
                    yield frame
                else:
                    num_bad_reads += 1
                    self.total_bad_reads += 1
            if not good_read:
                self.vid_cap.release()
                print("video capture closed")
            
    def __iter__(self):
        return self.video_generator()
    
    def __del__(self):
        if self.vid_cap.isOpened():
            self.vid_cap.release()
    
    def is_more_frames(self):
        return self.vid_cap.isOpened()
    
    def get_read_frame_info(self):
        print('{} frames have been read with {} bad reads'.format(self.total_frames_read,
                                                                  self.total_bad_reads))

In [6]:
#folder = './models'
folder = 'C:/Users/Edward/Dropbox/bats-code/models'

# model_filename = 'model_ThreeLayerWide_epochs_10_batcheff_4_lr_0.05_momentum_0.5_aug_aug-no-blur-2d-17Nov-big-dataset.tar'
# model_filename = 'model_UNET_epochs_100_batcheff_16_lr_0.01_momentum_0.9_aug_aug-2d-20Nov-big-dataset.tar'
# model_filename = 'model_UNET_epochs_100_batcheff_16_lr_0.01_momentum_0.9_aug_better-norm-aug-2d-20Nov-big-dataset.tar'
# model_file = os.path.join(folder, model_filename)
# model_file = './models/model_UNETTraditional_epochs_100_batcheff_16_lr_0.01_momentum_0.9_aug_better-norm-aug-2d-20Nov-big-dataset.tar'
# model_filename = 'model_UNET_epochs_100_batcheff_16_lr_0.01_momentum_0.9_aug_aug-2d-20Nov-big-dataset.tar'
# model_filename = 'model_UNET_epochs_100_batcheff_16_lr_0.01_momentum_0.9_aug_better-norm-aug-2d-20Nov-big-dataset.tar'
# model_filename = 'model_UNETTraditional_epochs_10_batcheff_16_lr_0.01_momentum_0.9_aug_better-norm-aug-2d-20Nov-big-dataset.tar'
model_filename = 'model_UNETTraditional_epochs_100_batcheff_16_lr_0.01_momentum_0.9_aug_better-norm-aug-2d-20Nov-big-dataset.tar'
#model_file = os.path.join(folder, model_filename)
model_file = os.path.join(folder.replace("\\", "/"), model_filename)

# device = torch.device("cuda")
# model = UNET(1, 2, should_pad=False)
# model.load_state_dict(torch.load(model_file))
# model.to(device)


In [7]:
#model = UNET(1, 2, should_pad=False)
#print(model.state_dict().keys())


In [8]:
#root_train_folder = ".../kasanka-bats/annotations"
root_train_folder = r"C:\Users\Edward\Dropbox\bats-code\data"
mean = np.load(os.path.join(root_train_folder, 'mean.npy'))
std = np.load(os.path.join(root_train_folder, 'std.npy'))

channel = 2
       
# augmentor = None
bat_datasets = []
for camera_folder in camera_folders:
    videos = sorted(glob.glob(os.path.join(camera_folder, '*.[Mm][Pp]4')))
    if not videos:
        continue
    # print(videos)
    augmentor = MaskCompose([Mask3dto2d(channel_to_use=channel),
                         MaskToTensor(),
                         MaskNormalize(mean[channel]/255, std[channel]/255),
                        ])
    bat_dataset = BatIterableDataset(videos, augmentor=augmentor)
    #print(camera_folder.split('\\')[-2:])
    save_folder = os.path.join(root_output_folder, *camera_folder.split('\\')[-2:])
    #print(save_folder)
    os.makedirs(save_folder, exist_ok=True)
    os.makedirs(os.path.join(save_folder, 'example-frames'), exist_ok=True)
    bat_datasets.append({'dataset':bat_dataset,
                         'save_folder': save_folder})
                                    

In [9]:
for i in bat_datasets[0]['dataset']:
    break
# dataloader = data.DataLoader(bat_dataset['dataset'], 
#                                  batch_size=batch_size,
#                                  shuffle=False, num_workers=0, 
#                                  pin_memory=True)

IndexError: list index out of range

In [ ]:
print(camera_folders)

In [ ]:
print(bat_datasets)

In [ ]:
plt.figure(figsize=(20,20))
# plt.imshow(((np.squeeze(i['image']) - mean) / std)[:,:,1])
plt.imshow(np.squeeze(i['image']))
plt.colorbar()
# plt.figure(figsize=(20,20))


In [ ]:
# save_folder
root_output_folder = 'H:\\kasanka-bats'
for bat_dataset in bat_datasets[:]:
    normalized_path = os.path.normpath(bat_dataset['save_folder'])
    day = normalized_path.split(os.sep)[-2]
    location = normalized_path.split(os.sep)[-1]
    save_folder = os.path.join(root_output_folder, day, location)
    print(save_folder)
    # print(os.path.join(save_folder, 'size.npy'))

In [ ]:
def logit2prob(logit):
    e_l = np.e ** logit
    return e_l 

def denorm_image(im, mean, std):
    """ Take image the was normalized and return to 0 to 255"""
#     im = np.copy(im)
    im *= std
    im += mean
    im *= 255
    im = np.maximum(im, 0)
    im = np.minimum(im, 255)
    im = im.astype(np.uint8)
    
    return im

should_plot = False
should_save = True

num_classes = 2
bat_prob_thresh = .6
batch_size = 1
#2
early_stop = None
# save some original frames to check detection quality
save_every_n_frames = 1350
channel = 2


device = torch.device("cuda")
model = UNETTraditional(1, 2, should_pad=False)
model.load_state_dict(torch.load(model_file))
model.to(device)

model.train(False)

for bat_dataset in bat_datasets[:]: #reversed(bat_datasets[:]): #

    num_frames = 0
    running_loss = 0
    
    print(bat_dataset['save_folder'])
    
    dataloader = data.DataLoader(bat_dataset['dataset'], 
                                 batch_size=batch_size,
                                 shuffle=False, num_workers=0, 
                                 pin_memory=True)

    centers_list = []
    sizes_list = []
    rects_list = []
    contours_list = []  # Add this list for storing processed contours


    for batch_ind, batch in enumerate(dataloader):
        if batch_ind == 0:
            print('started...')
            t0 = time.time()
        if early_stop:
            if batch_ind >= early_stop:
                break

        im_batch = batch['image'].cuda()
    #     masks = batch['mask'].cuda()

        with torch.no_grad():
            outputs = model(im_batch)
            masks = (outputs[:, 1].cpu().numpy() > np.log(bat_prob_thresh)).astype(np.uint8)
            
            for ind, mask in enumerate(masks):
                centers, areas, contours, _, _, rects = bat_functions.get_blob_info(mask)
                centers_list.append(centers)
                sizes_list.append(areas)
                contours_list.append(contours)
                rects_list.append(rects)
                if save_every_n_frames:
                    if num_frames % save_every_n_frames == 0:
                        print(bat_dataset['save_folder'])
                        normalized_path = os.path.normpath(bat_dataset['save_folder'])
                        day = normalized_path.split(os.sep)[-2]
                        location = normalized_path.split(os.sep)[-1]
                        save_folder = os.path.join(root_output_folder, day, location, "example-frames")
                        im_name = '{}_{}_obs-ind-{}.jpg'.format(day, location, num_frames)
                        im_file = os.path.join(save_folder, im_name)
                        print(im_file)
                        im = np.squeeze(batch['image'][ind].numpy())
                        im = denorm_image(im, mean[channel]/255, std[channel]/255)
                        cv2.imwrite(im_file, im)
                num_frames += 1


        if should_plot:
            for ind in range(len(im_batch)):

                if 'orig' in batch.keys():
                    plt.figure(figsize=(10,10))
                    plt.imshow(batch['orig'][ind])
                plt.figure(figsize=(10,10))
                im = im_batch[ind].cpu().numpy()
                im = np.transpose(im, (1, 2, 0))
                plt.imshow(im)
                plt.figure(figsize=(10,10))
                im = outputs[ind][0].cpu().numpy()
                plt.imshow(im)
                plt.title('output')
                prob = logit2prob(outputs[ind,1].cpu().numpy())
                mask = (prob > 0.5).astype(np.uint8)

    #             display_im = np.zeros_like(im)
    #             display_im[..., 0] = masks[ind]
                plt.figure(figsize=(10,10))
                plt.imshow(mask)
    #             plt.colorbar()
    #             plt.figure(figsize=(10,10))
    #             plt.imshow(display_im)

    total_time = time.time() - t0
    print(total_time, total_time / batch_ind / batch_size)
    print(bat_dataset['dataset'].get_read_frame_info())
    if should_save:
        normalized_path = os.path.normpath(bat_dataset['save_folder'])
        day = normalized_path.split(os.sep)[-2]
        location = normalized_path.split(os.sep)[-1]
        save_folder = os.path.join(root_output_folder, day, location)
        # save_folder = bat_dataset['save_folder']
        num_contour_files = 15
        file_num = 0
        new_contours = []
        for frame_ind, cs in enumerate(contours_list):
            if frame_ind % int(len(contours_list)/num_contour_files) == 0:
                # start new file
                file_name = f'contours-compressed-{file_num:02d}.npy'
                file = os.path.join(save_folder, file_name)
                np.save(file, np.array(new_contours, dtype=object))
                new_contours = []
                file_num += 1
            new_contours.append([])
            for c in cs:
                cc	= np.squeeze(cv2.approxPolyDP(c, 0.1, closed=True))
                new_contours[-1].append(cc)
        file_name = f'contours-compressed-{file_num:02d}.npy'
        file = os.path.join(save_folder, file_name)
        np.save(file, np.array(new_contours, dtype=object))
        #np.save(os.path.join(save_folder, 'size.npy'), sizes_list, dtype = object)
        #np.save(os.path.join(save_folder,'rects.npy'), rects_list)
        #np.save(os.path.join(save_folder, 'centers.npy'), centers_list)
        #np.save(os.path.join(save_folder, 'flipped_centers.npy'), centers_list)
        with open(os.path.join(save_folder, 'size.npy'), 'wb') as f:
            pickle.dump(sizes_list, f)

        # To save rects_list
        with open(os.path.join(save_folder, 'rects.npy'), 'wb') as f:
            pickle.dump(rects_list, f)

        # To save centers_list
        with open(os.path.join(save_folder, 'flipped_centers.npy'), 'wb') as f:
            pickle.dump(centers_list, f)

# latest error 
4/9/23: ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (49467,) + inhomogeneous part.
20221219\3 Chinyangali could not write size.npy file

## previous error 
3/9/23: RuntimeError: cuDNN error: CUDNN_STATUS_EXECUTION_FAILED

In [ ]:
def get_list_structure(lst):
    if not isinstance(lst, list):
        return "Not a list"

    structure = []

    def traverse(sublist):
        if isinstance(sublist, list):
            structure.append(len(sublist))
            if sublist:
                traverse(sublist[0])
        else:
            structure.append("Element")

    traverse(lst)
    return structure
print(get_list_structure(new_contours))
print(get_list_structure(sizes_list))
print(get_list_structure(rects_list))
print(get_list_structure(centers_list))

In [ ]:
# Convert sizes_list to a NumPy array
sizes_array = np.array(sizes_list, dtype="object")
print(sizes_array)

In [ ]:
#for item in sizes_list:
#    if not isinstance(item, int):
#        print(f"Found a non-integer: {item}, type: {type(item)}")

In [ ]:
## Filter out non-integer elements
#filtered_sizes_list = [item for item in sizes_list if isinstance(item, int)]

## Convert the filtered list to a NumPy array
#sizes_array = np.array(filtered_sizes_list, dtype=np.int32)
#print(sizes_array)

In [ ]:
# print(os.sep)

In [ ]:
# D:\kasanka-bats\221101 Bat Count\9 KK
torch.cuda.memory_stats()

In [ ]:
torch.cuda.memory_summary()

In [ ]:
torch.cuda.empty_cache()

In [ ]:
torch.cuda.memory_stats()

In [ ]:
# pip install GPUtil

In [ ]:
import GPUtil
GPUtil.showUtilization()

In [ ]:
torch.cuda.empy_cache()

In [ ]:
import torch
from GPUtil import showUtilization as gpu_usage

print("Initial GPU Usage")
gpu_usage()                             

tensorList = []
for x in range(10):
  tensorList.append(torch.randn(10000000,10).cuda())   # reduce the size of tensor if you are getting OOM
  
  

print("GPU Usage after allcoating a bunch of Tensors")
gpu_usage()

del tensorList

print("GPU Usage after deleting the Tensors")
gpu_usage()  

print("GPU Usage after emptying the cache")
torch.cuda.empty_cache()
gpu_usage()